# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/imatiq/ML_Internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

Unit of analysis: one row = one content page, described by a 90-day rolling snapshot as of
the starter dataset's export (content_refresh_anonymized.csv). This is a single-snapshot
dataset — it does NOT contain repeated daily observations per page like the warehouse's
fact_content_daily_performance table does. Each row is one page's state at one point in time,
not a time series.

In [6]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/imatiq/ML_Internship"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
else:
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

print("Working dir:", os.getcwd())
assert os.path.exists("data/raw/content_refresh_anonymized.csv"), "starter CSV not found — are you at the repo root?"
print("Starter data found. You're ready.")

Working dir: /content/flyrank-ml-internship-starter/flyrank-ml-internship-starter
Starter data found. You're ready.


In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

print(f"Rows: {len(df):,}")
print(f"Unique content_id: {df['content_id'].nunique():,}  <- confirms one row per page (grain check)")
print(f"Unique client_id: {df['client_id'].nunique():,}")
print(f"content_age_days range: {df['content_age_days'].min()} to {df['content_age_days'].max()} days")

Rows: 30,000
Unique content_id: 30,000  <- confirms one row per page (grain check)
Unique client_id: 32
content_age_days range: 90 to 564 days


## 2. Fields: feature / label / context / excluded

Features (used for clustering):
impressions_90d, ctr, avg_position, engagement_rate, scroll_rate, word_count,
content_age_days, days_since_last_update, ai_traffic_pct, competition, cpc

Label/proxy: None — this is unsupervised. Cluster membership is generated by the algorithm,
not read from an existing column.

Context (used for grouping/inspection, never as a clustering feature):
content_id, client_id — join keys only, carry no signal about performance.

Excluded, with why:
- trend_direction, trend_pct: these are derived FROM the same 90-day window I'm clustering
  on, and trend_direction is literally the label used elsewhere (Notebook 01-02) for
  is_declining. Including them would let the cluster axis be dominated by a single
  precomputed judgment rather than raw observable behavior.
- position_tier, impression_tier, freshness_tier, age_tier, char_count_tier: these are
  bucketed versions of numeric fields I already include (avg_position, impressions_90d,
  etc.). Including both the raw number and its bucket double-counts the same signal.
- search_volume: Notebook 01 showed near-zero correlation (0.001) with actual traffic —
  keeping it would add noise without adding real signal to the clustering distance.

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
features = ["impressions_90d", "ctr", "avg_position", "engagement_rate", "scroll_rate",
            "word_count", "content_age_days", "days_since_last_update", "ai_traffic_pct",
            "competition", "cpc"]
excluded = ["trend_direction", "trend_pct", "position_tier", "impression_tier",
            "freshness_tier", "age_tier", "char_count_tier", "search_volume"]

print("Features:", len(features), "| Excluded:", len(excluded))
print("All features exist in df:", all(f in df.columns for f in features))

Features: 11 | Excluded: 8
All features exist in df: True


## 3. Verify it with queries (grain, counts, missing values, windows)

Every claim above, checked against the actual data:

In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Grain: exactly one row per content_id (no duplicates)
print("Duplicate content_id rows:", df["content_id"].duplicated().sum())

# Missing values in chosen features
print("\nMissing values per feature:")
print(df[features].isna().sum())

# Window: confirm impressions_90d and engagement metrics are non-negative, sane ranges
print("\nFeature ranges:")
print(df[features].describe().T[["min", "max"]])

Duplicate content_id rows: 0

Missing values per feature:
impressions_90d              0
ctr                          0
avg_position                 0
engagement_rate              0
scroll_rate                125
word_count                7699
content_age_days             0
days_since_last_update       0
ai_traffic_pct               0
competition               2468
cpc                       2468
dtype: int64

Feature ranges:
                         min        max
impressions_90d          1.0  517715.00
ctr                      0.0     100.00
avg_position             0.0     245.00
engagement_rate          0.0     100.00
scroll_rate              0.0     300.00
word_count               8.0    9546.00
content_age_days        90.0     564.00
days_since_last_update   1.0     373.00
ai_traffic_pct           0.0     300.00
competition              0.0       1.00
cpc                      0.0     100.36


## 4. Data limits

What this data can never tell me:

1. No time dimension per page: this is a single snapshot, so I cannot cluster on HOW a
   page's behavior changes over time — only its current state. Any "momentum" or "trend"
   archetype would need the warehouse's daily fact table instead.

2. Client imbalance: some clients likely contribute far more pages than others. A cluster
   could end up reflecting one client's content style rather than a genuine cross-client
   archetype. I need to check page counts per client before trusting any cluster's story.

3. No causal claims: clustering can show that a group of pages shares similar metrics —
   it cannot show that being in that group CAUSES good or bad performance.

4. Missing values: word_count (and possibly others) has NaN rows (seen in w02) — these
   need an explicit imputation or exclusion decision before clustering, or the algorithm
   will silently drop or mishandle them.

In [10]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Client imbalance check
pages_per_client = df["client_id"].value_counts()
print("Pages per client — top 5:")
print(pages_per_client.head())
print(f"\nMedian pages per client: {pages_per_client.median():.0f}")
print(f"Max pages from a single client: {pages_per_client.max()}")
print(f"Share of dataset from top client: {pages_per_client.max()/len(df):.1%}")

Pages per client — top 5:
client_id
client_19581e27de    7008
client_6208ef0f77    3681
client_4e07408562    2294
client_3fdba35f04    2267
client_f369cb89fc    1796
Name: count, dtype: int64

Median pages per client: 567
Max pages from a single client: 7008
Share of dataset from top client: 23.4%


## Self-check

Before you submit, confirm each line honestly:

- [X] Every section above is filled — markdown thinking AND the code that backs it
- [X] The notebook runs top to bottom with no errors (Runtime → Run all)
- [X] No client names, URLs, or private queries anywhere
- [X] My claims use careful words: observed, measured, directional, decision-support
- [X] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.